# The Case for RAG — Why Language Models Need Retrieval

**Notebook 01 · Phase 1 (Conceptual)** · Stack: `langchain` + `langchain-openai` + OpenAI

This notebook does one thing: it **demonstrates, with live model calls, the three
failure modes that motivate Retrieval-Augmented Generation (RAG)** — and then shows
how adding a retrieval step fixes them.

We are *not* building a vector database or embeddings yet (that is Phase 2). Here we
prove *why* retrieval is necessary in the first place, using the simplest possible
"retriever": handing the model the right document at inference time.

---

## The problem: an LLM is a closed book

A large language model answers from **parametric memory** — everything it learned
during training, frozen into its weights. That produces three predictable failures:

| # | Failure mode | What it looks like |
|---|--------------|--------------------|
| 1 | **Hallucination under knowledge gaps** | The model generates *fluent but false* answers for things it was never taught. It doesn't know that it doesn't know. |
| 2 | **Knowledge cutoff** | The model's world-knowledge is *frozen* at its training date. Anything after that is invisible. |
| 3 | **Private / enterprise data** | The model has never seen your internal wiki, contracts, claims data, or support tickets — so it cannot answer questions about them. |

## The fix: RAG makes it an open book

RAG retrieves relevant documents from *your* corpus and places them in the model's
context **at inference time**. The model reads before it writes. This delivers the
**three promises of RAG**:

- 🎯 **Groundedness** — answers are generated *from evidence*, not from memory.
- 🕒 **Freshness** — new documents are available immediately; no retraining needed.
- 🔖 **Traceability** — every answer can cite the source it came from.

Let's prove each of these.

## 0. Install dependencies

Run this cell **first**. It installs everything this notebook needs into the *active
kernel* using `%pip`, so it works in any fresh environment — if a package is already
present, pip simply skips it. **After the very first install, restart the kernel**
(Kernel → Restart Kernel) so newly installed packages are picked up, then run the rest
top to bottom.

> `numpy` is pinned `<2` because the OCR stack (opencv / rapidocr) needs it.

In [1]:
# Install dependencies into the ACTIVE kernel (idempotent — skips what's already there).
%pip install -q \
    langchain langchain-core langchain-openai \
    python-dotenv
print("✅ Dependencies ready. If this was a fresh install, restart the kernel now, then re-run.")


[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: /Users/mohamednoordeenalaudeen/Documents/GenAI-2026/.venv/bin/python -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.
✅ Dependencies ready. If this was a fresh install, restart the kernel now, then re-run.


## 1. Setup

We load API keys from the `.env` file in the project root (one directory up).
`python-dotenv` reads the file; we **never print the key values**.

> If `langchain-openai` is not installed, run: `pip install langchain-openai python-dotenv`

In [2]:
from pathlib import Path
from dotenv import load_dotenv
import os

# The .env lives in the project root, one level above this notebooks/ folder.
env_path = Path.cwd().parent / ".env"
loaded = load_dotenv(env_path)

# Verify the key is present WITHOUT revealing it.
key = os.getenv("OPENAI_API_KEY")
print(f".env found & loaded : {loaded}")
print(f"OPENAI_API_KEY set  : {bool(key)}")
print(f"Key preview         : {key[:7] + '...' + key[-4:] if key else 'MISSING'}")

.env found & loaded : True
OPENAI_API_KEY set  : True
Key preview         : sk-proj...QKAA


In [3]:
from langchain_openai import ChatOpenAI

# temperature=0 makes the demos as reproducible as possible.
# gpt-4o-mini is cheap and fast — perfect for teaching.
MODEL = "gpt-4o-mini"
llm = ChatOpenAI(model=MODEL, temperature=0)

print(f"Chat model ready: {MODEL}")

Chat model ready: gpt-4o-mini


### A tiny helper

`ask()` sends a plain question to the model with **no retrieved context** — this is
the "closed book" baseline we want to expose. We use a LangChain Expression Language
(LCEL) chain: `prompt | llm | parser`.

In [4]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# A deliberately "confident assistant" system prompt — it forbids hedging and
# demands a specific answer. This is how naive LLM apps are often wired (to avoid
# "unhelpful" refusals), and it is exactly what surfaces hallucination.
closed_book_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "You are a knowledgeable expert assistant. Always give a specific, confident, "
     "direct answer using concrete details (names, numbers, dates). Do NOT refuse, "
     "do NOT say you lack access to documents, and do NOT tell the user to check "
     "elsewhere. Answer to the best of your knowledge."),
    ("human", "{question}"),
])

closed_book = closed_book_prompt | llm | StrOutputParser()

def ask(question: str) -> str:
    answer = closed_book.invoke({"question": question})
    print(answer)
    return answer

## 2. Failure mode #1 — Hallucination under knowledge gaps

When a question falls outside what a model learned in training, it does not stop —
it produces a *fluent, confident, and false* answer. Below are **three proofs**,
each probing a different kind of knowledge gap. Watch how specific and self-assured
the fabrications are.

> We phrase these as plain general-knowledge questions, *not* "quote document X."
> Asking a model to quote a named document triggers a trained "I can't access
> documents" refusal — a different behavior from the hallucination we want to expose.
> (Temperature is 0, so each run is reproducible; a future model may occasionally
> *correct* one of these instead of fabricating — that's informative too.)

### Proof 1 — Private / enterprise data

A company we invented ("Northwind Logistics"). There is no public answer, so any
specific figure the model gives is manufactured.

In [5]:
_ = ask(
    "What is the monthly home-internet stipend amount for remote employees at "
    "Northwind Logistics, and who is eligible for it?"
)

Northwind Logistics provides a monthly home-internet stipend of $75 for remote employees. This stipend is available to all remote employees who are required to work from home as part of their job responsibilities. Eligibility typically includes full-time remote workers and may extend to part-time remote employees depending on their specific roles and needs.


### Proof 2 — A plausible-but-fake product

The "Sony Xperia Z9 Ultra" does not exist. But the name *sounds* real, so the model
happily invents a full spec sheet — processor, display, battery, release date.

In [6]:
_ = ask(
    "What are the full technical specifications of the Sony Xperia Z9 Ultra "
    "smartphone — its processor, display size, battery capacity, and release date?"
)

The Sony Xperia Z9 Ultra was officially announced on September 3, 2015, and it features the following technical specifications:

- **Processor**: Qualcomm Snapdragon 810, an octa-core processor with a clock speed of up to 2.0 GHz.
- **Display**: 6.44-inch IPS LCD display with a resolution of 2560 x 1440 pixels (Quad HD), offering a pixel density of approximately 342 ppi.
- **Battery Capacity**: 3000 mAh non-removable battery, providing decent longevity for the device.
  
These specifications highlight the Xperia Z9 Ultra as a high-end smartphone for its time, focusing on performance and display quality.


### Proof 3 — A fabricated academic citation

A paper that was never written. The model will confidently summarize "findings,"
inventing a citation out of thin air — the exact behavior that got a lawyer
sanctioned for filing non-existent cases.

In [7]:
_ = ask(
    "Summarize the key findings of the 2021 research paper 'Retrieval Beats "
    "Memory: A Study of Grounded Language Models' by Chen and Alvarez."
)

The 2021 research paper "Retrieval Beats Memory: A Study of Grounded Language Models" by Chen and Alvarez investigates the effectiveness of retrieval-augmented approaches in language models compared to traditional memory-based methods. The key findings of the study include:

1. **Retrieval-Augmented Performance**: The authors demonstrate that models that incorporate retrieval mechanisms outperform those relying solely on memory. This suggests that accessing external information can enhance the model's ability to generate accurate and contextually relevant responses.

2. **Grounded Language Models**: The study emphasizes the importance of grounding language models in external knowledge sources. By integrating retrieval systems, models can leverage vast amounts of information, leading to improved performance on various language tasks.

3. **Task-Specific Advantages**: The research highlights that retrieval-augmented models show significant advantages in tasks requiring factual knowledge 

**What just happened?** Three questions, three confident fabrications — a made-up
stipend, a spec sheet for a phone that was never built, and findings from a paper
that does not exist.

> This is the dangerous part: *the model doesn't know that it's wrong.* Nothing
> inside it separates a recalled fact from a plausible invention. At production
> scale this is how chatbots invent refund policies and lawyers cite fictitious
> cases. The **real** Northwind policy ($45/month, clause HR-2024-17 §7.3) lives
> only in our private docs — we'll retrieve it in Section 5 and watch the answer
> change from fiction to a cited fact.

## 3. Failure mode #2 — Knowledge cutoff

The model's world-knowledge is **frozen** at its training cutoff. Let's ask it to
tell us its own cutoff, and then ask about something that changes constantly.

In [8]:
_ = ask("What is your training-data knowledge cutoff date? Answer in one sentence.")

My training data knowledge cutoff date is October 2023.


In [9]:
_ = ask(
    "Who won the most recent FIFA World Cup final, on what date, and what was the "
    "exact final score including any penalty shootout?"
)

The most recent FIFA World Cup final took place on December 18, 2022. Argentina won the match against France with a final score of 3-3 after extra time, and Argentina triumphed 4-2 in the penalty shootout.


**What just happened?** The model answers from a frozen snapshot of the world. For
anything that happened after its cutoff — a match result, a product launch, a stock
price, a newly released model — it is either silent, outdated, or (worse) confidently
fills the gap with a guess. Retraining a model to fix this costs millions and takes
months. **RAG injects fresh information at inference time instead.**

## 4. Failure mode #3 — Private / enterprise data

Even a perfectly up-to-date model has **never seen your internal data** — your wiki,
contracts, claims records, or support tickets live behind your VPN. Web search does
not fix this. Let's ask about an internal decision.

In [10]:
_ = ask(
    "In our internal 'Project Aurora' engineering sync on Monday, which vector "
    "database did the team decide to standardize on for the claims-search service, "
    "and what was the main reason given?"
)

In your internal 'Project Aurora' engineering sync on Monday, the team decided to standardize on Pinecone as the vector database for the claims-search service. The main reason given for this decision was Pinecone's scalability and performance in handling high-dimensional vector data, which is crucial for efficiently processing and retrieving claims-related information. Additionally, its ease of integration with existing systems and robust support for real-time updates were significant factors in the decision.


**What just happened?** "Project Aurora" is our private context. The model has two
options — refuse, or invent a plausible-sounding summary. Neither is useful, because
the *actual* answer only exists inside our organization. This is the single biggest
reason enterprises need RAG: **your knowledge is invisible to every off-the-shelf LLM.**

## 5. The fix — Retrieval-Augmented Generation

Now we give the model an **open book**. The pattern has three stages:

```
        ┌──────────┐     ┌──────────┐     ┌──────────┐
Query → │ RETRIEVE │  →  │ AUGMENT  │  →  │ GENERATE │ → Answer + Source
        └──────────┘     └──────────┘     └──────────┘
```

- **Retrieve** — find the documents relevant to the question.
- **Augment** — insert those documents into the prompt as context.
- **Generate** — the model answers *from that context*, and cites it.

For Phase 1 we use a **tiny, transparent keyword retriever** over an in-memory
"knowledge base." In Phase 2 this retriever is replaced by embeddings + a vector
database — but the RAG *contract* is identical.

In [11]:
# A minimal private knowledge base: (doc_id, text). In reality these are chunks
# pulled from your wiki, contracts, tickets, etc.
KNOWLEDGE_BASE = [
    ("HR-2024-17 §7.3",
     "Northwind Logistics Remote-Work Reimbursement Policy (HR-2024-17), clause 7.3: "
     "Employees working remotely at least 3 days per week are eligible for a home-internet "
     "stipend of $45 per month, reimbursed via the monthly expense report. Contractors are "
     "not eligible."),
    ("Project Aurora — sync notes 2024-06-10",
     "Project Aurora engineering sync (Mon 2024-06-10): the team standardized on Qdrant "
     "for the claims-search service. Main reason: best-in-class metadata payload filtering "
     "under ~50M vectors, which the claims ACL model requires. pgvector was considered but "
     "rejected because per-tenant filtering at our recall target was slower in testing."),
    ("Onboarding FAQ",
     "New engineers get laptop provisioning within 2 business days and VPN access on day one."),
]

def retrieve(query: str, k: int = 2):
    """Toy retriever: score docs by keyword overlap. Returns top-k (id, text)."""
    q_terms = {w.strip(".,?()").lower() for w in query.split() if len(w) > 3}
    scored = []
    for doc_id, text in KNOWLEDGE_BASE:
        doc_terms = {w.strip(".,?()").lower() for w in text.split()}
        overlap = len(q_terms & doc_terms)
        scored.append((overlap, doc_id, text))
    scored.sort(reverse=True)
    return [(doc_id, text) for score, doc_id, text in scored[:k] if score > 0]

# Sanity check
for doc_id, _ in retrieve("Project Aurora vector database decision"):
    print("retrieved:", doc_id)

retrieved: Project Aurora — sync notes 2024-06-10


### The grounded chain

The system prompt now enforces the **three promises**: answer **only** from the
provided context (groundedness), and **cite the source id** for every claim
(traceability). If the context lacks the answer, say so — no more fabrication.

In [12]:
rag_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "You are a grounded assistant. Use ONLY the information in the CONTEXT to answer. "
     "Cite the source id in square brackets after each fact, e.g. [HR-2024-17 §7.3]. "
     "If the answer is not in the context, reply exactly: "
     "\"I don't have enough information to answer that.\""),
    ("human", "CONTEXT:\n{context}\n\nQUESTION: {question}"),
])

rag_chain = rag_prompt | llm | StrOutputParser()

def ask_rag(question: str, k: int = 2) -> str:
    docs = retrieve(question, k=k)
    context = "\n\n".join(f"[{doc_id}] {text}" for doc_id, text in docs)
    print("--- Retrieved sources ---")
    for doc_id, _ in docs:
        print(" •", doc_id)
    print("\n--- Grounded answer ---")
    answer = rag_chain.invoke({"context": context, "question": question})
    print(answer)
    return answer

### Groundedness + Traceability

Re-ask the **private-data** question — the one the closed-book model fabricated in
Section 4. This time the answer comes from the retrieved note and carries a citation.

In [13]:
_ = ask_rag(
    "In our internal Project Aurora sync, which vector database did the team "
    "standardize on for the claims-search service, and why?"
)

--- Retrieved sources ---
 • Project Aurora — sync notes 2024-06-10

--- Grounded answer ---


The team standardized on Qdrant for the claims-search service because it offers best-in-class metadata payload filtering under ~50M vectors, which is required by the claims ACL model. Pgvector was considered but rejected due to slower per-tenant filtering at the recall target during testing [Project Aurora — sync notes 2024-06-10].


### Before / after on the *same* question

Recall Section 2, where the closed-book model invented a stipend amount. Here is
the identical question answered **with retrieval** — grounded in the real policy
document, and cited.

In [14]:
# The same stipend question that was fabricated closed-book in Section 2.
_ = ask_rag(
    "What is the monthly home-internet stipend for remote employees at "
    "Northwind Logistics, and who is eligible?"
)

--- Retrieved sources ---
 • HR-2024-17 §7.3

--- Grounded answer ---


The monthly home-internet stipend for remote employees at Northwind Logistics is $45. Employees working remotely at least 3 days per week are eligible for this stipend, while contractors are not eligible [HR-2024-17 §7.3].


### Honesty when the answer is absent

RAG also fixes hallucination in the *other* direction: when the corpus genuinely
doesn't contain the answer, a grounded system **admits it** instead of inventing.

In [15]:
_ = ask_rag("What is Northwind Logistics' parental-leave policy?")

--- Retrieved sources ---
 • HR-2024-17 §7.3

--- Grounded answer ---


I don't have enough information to answer that.


### Freshness

No retraining required. We just **add a new document** to the knowledge base, and the
system can answer about it on the very next query — information that was impossible a
moment ago.

In [16]:
KNOWLEDGE_BASE.append(
    ("Incident INC-5521 — 2024-06-14",
     "On 2024-06-14 the claims-search service had a 22-minute outage caused by a Qdrant "
     "snapshot restore that exhausted disk. Fix: increased volume size and added a disk-usage "
     "alert at 80%. Owner: platform team.")
)

_ = ask_rag("What caused incident INC-5521 and how was it fixed?")

--- Retrieved sources ---
 • Incident INC-5521 — 2024-06-14

--- Grounded answer ---


Incident INC-5521 was caused by a Qdrant snapshot restore that exhausted disk, leading to a 22-minute outage of the claims-search service. It was fixed by increasing the volume size and adding a disk-usage alert at 80% [Incident INC-5521 — 2024-06-14].


## 6. Summary — closed book vs. open book

| Question type | Closed-book LLM (Sections 2–4) | With RAG (Section 5) |
|---------------|-------------------------------|----------------------|
| Fake/unknown policy | Fabricates confidently ❌ | Cites real clause, or says "not enough info" ✅ |
| Post-cutoff event | Outdated or guessed ❌ | Answer from freshly added doc ✅ |
| Private "Project Aurora" | Invents a plausible summary ❌ | Grounded answer **with citation** ✅ |

### The three promises, proven

- 🎯 **Groundedness** — the grounded chain answered *from retrieved evidence*, and
  said "I don't have enough information" when the corpus was silent.
- 🕒 **Freshness** — appending a document made new facts answerable instantly, with
  no retraining.
- 🔖 **Traceability** — every grounded answer carried a `[source id]` citation.

### Where this goes next (Phase 2)

The only toy part here is `retrieve()` — keyword overlap over an in-memory list. In
production you replace it with:

1. **Chunking** documents sensibly (see the chunking guide in `reference/`).
2. **Embeddings** to capture meaning, not keywords.
3. A **vector database** (pgvector, Qdrant, Milvus…) for fast semantic search at scale.
4. Optionally a **reranker** to sharpen precision before generation.

The `retrieve → augment → generate` contract — and the three promises — stay exactly
the same.